In [27]:
import pandas as pd
import numpy as np

import os
from sqlalchemy import create_engine
from datetime import datetime



In [28]:
hostname = "AdhamNourMainPC"
dbname = "stg_personal_financial_analysis"
uname = "root"
pwd = "production_server"

In [29]:
file_path=r"C:\Users\adham\OneDrive\Documents\Personal\Finance\Banking\CIB\Credit Cards\Platinum_202501.xls"

In [30]:
df=pd.read_excel(io=file_path)

WARNING *** file size (265760) not 512 + multiple of sector size (512)


In [31]:
index = df[df['Unnamed: 15'] == 'OPENING BALANCE'].index[0]+1
df=df.loc[index:]

In [32]:
index = df[df['Unnamed: 15'] == 'CLOSING BALANCE'].index[0]-1
df=df.loc[:index]


In [33]:
df = df.iloc[:, [4, 8, 15, 33]]
df

,Unnamed: 4,Unnamed: 8,Unnamed: 15,Unnamed: 33
27,NaN,NaN,NaN,NaN
28,24-12,25-12,EPP CAN-Balance Conversion Debit,2460.01
29,NaN,NaN,NaN,NaN
30,NaN,NaN,Card No: 5229 XXXX XXXX 3527,NaN
31,NaN,NaN,NaN,NaN
...,...,...,...,...
440,NaN,NaN,NaN,NaN
441,20-01,21-01,Month.Interest,538.54
442,NaN,NaN,NaN,NaN
443,24-12,01-12,Balance Conversion Credit,2460.01 CR


In [34]:
df.columns = ["Transaction_Date","Value_Date","Description","Amount"]

In [35]:
df

,Transaction_Date,Value_Date,Description,Amount
27,NaN,NaN,NaN,NaN
28,24-12,25-12,EPP CAN-Balance Conversion Debit,2460.01
29,NaN,NaN,NaN,NaN
30,NaN,NaN,Card No: 5229 XXXX XXXX 3527,NaN
31,NaN,NaN,NaN,NaN
...,...,...,...,...
440,NaN,NaN,NaN,NaN
441,20-01,21-01,Month.Interest,538.54
442,NaN,NaN,NaN,NaN
443,24-12,01-12,Balance Conversion Credit,2460.01 CR


In [36]:
for column in df.columns:
    if column != 'Amount':
        df[column] = df[column].fillna(method='ffill')


C:\Users\adham\AppData\Local\Temp\ipykernel_33136\3942618212.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='ffill')


In [37]:
df = df[df['Amount'].notna()]


In [38]:
df['Transaction_Date'].replace('00-00',np.nan,inplace=True)
df['Value_Date'].replace('00-00',np.nan,inplace=True)

C:\Users\adham\AppData\Local\Temp\ipykernel_33136\2919262837.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Transaction_Date'].replace('00-00',np.nan,inplace=True)
C:\Users\adham\AppData\Local\Temp\ipykernel_33136\2919262837.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Transaction_Date'].replace('00-00',np.nan,inplace=True)

In [39]:
df.to_excel(r"Platinum_202501_cleaned.xlsx", index=False)


In [40]:
df['Amount']=df['Amount'].astype('str').str.strip()


C:\Users\adham\AppData\Local\Temp\ipykernel_33136\116131844.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Amount']=df['Amount'].astype('str').str.strip()


In [41]:
df[['Transaction_Amount', 'sign']] = df['Amount'].apply(
    lambda x: x.split(" ", 1) if " " in x else [x, None]
).apply(pd.Series)
df

C:\Users\adham\AppData\Local\Temp\ipykernel_33136\2647094270.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[['Transaction_Amount', 'sign']] = df['Amount'].apply(
C:\Users\adham\AppData\Local\Temp\ipykernel_33136\2647094270.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[['Transaction_Amount', 'sign']] = df['Amount'].apply(


,Transaction_Date,Value_Date,Description,Amount,Transaction_Amount,sign
28,24-12,25-12,EPP CAN-Balance Conversion Debit,2460.01,2460.01,None
32,20-12,23-12,Instashop Cairo,641.35,641.35,None
34,28-12,02-01,SAHL GIZA,2065.45,2065.45,None
36,28-12,02-01,SAHL GIZA,55.70,55.70,None
38,01-01,05-01,VODAFONE TOP UP APP GIZA,236.00,236.00,None
...,...,...,...,...,...,...
435,05-01,07-01,FOREIGN EXCHANGE FEES,3.74,3.74,None
437,14-01,15-01,FOREIGN EXCHANGE FEES,5.99,5.99,None
439,16-01,19-01,LOAD WALLET FEES,5.00,5.00,None
441,20-01,21-01,Month.Interest,538.54,538.54,None


In [42]:
df['Transaction_Amount']=df['Transaction_Amount'].astype('float')
df['signed_amount'] = df['Transaction_Amount'].where(~df['sign'].isnull(), -1 * df['Transaction_Amount'])


C:\Users\adham\AppData\Local\Temp\ipykernel_33136\3204910101.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Transaction_Amount']=df['Transaction_Amount'].astype('float')
C:\Users\adham\AppData\Local\Temp\ipykernel_33136\3204910101.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['signed_amount'] = df['Transaction_Amount'].where(~df['sign'].isnull(), -1 * df['Transaction_Amount'])


In [43]:
df = df.drop(columns=['Amount', 'Transaction_Amount', 'sign'])

In [44]:
df['Transaction_Date'] = df['Transaction_Date'].fillna(method='ffill')
df['Value_Date'] = df['Value_Date'].fillna(method='ffill')

C:\Users\adham\AppData\Local\Temp\ipykernel_33136\300869884.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Transaction_Date'] = df['Transaction_Date'].fillna(method='ffill')
C:\Users\adham\AppData\Local\Temp\ipykernel_33136\300869884.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Value_Date'] = df['Value_Date'].fillna(method='ffill')


In [45]:
df['currency_value'] = df['Description'].shift(-1)
df['signed_ammount_shift'] = df['signed_amount'].shift(-1)





In [46]:
# Condition: signed_amont + signed_ammount_shift == signed_amont
mask = df['signed_amount'] + df['signed_ammount_shift'] != df['signed_amount']

# Update "Description" column by appending 'currency_value'
df.loc[mask, 'currency_value'] = np.nan


In [50]:
df['currency']=df['currency_value'].astype('str').str[:3].replace('nan',np.nan);
df['value']=df['currency_value'].astype('str').str[3:].replace('nan',np.nan);


In [51]:
df.to_excel(r"Platinum_202501_cleaned.xlsx", index=False)


In [ ]:
filename = os.path.basename(file_path)

df['file_name']=filename


In [ ]:
df


In [ ]:
engine = create_engine(f"mysql+pymysql://{uname}:{pwd}@{hostname}/{dbname}")

In [ ]:
df.to_sql("credit_card_transaction_files", engine, if_exists="replace", index=False)